<a href="https://colab.research.google.com/github/isaacadebayo/Predictive-Analytics-Public-Datasets/blob/main/Apple_Stock_prediction_timeseries_split_fold4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What is MLflow?

MLflow is an open-source platform for managing the end-to-end machine learning lifecycle. It addresses four primary functions:

1.  **Tracking:** Record and query experiments (code, data, configuration, results).
2.  **Projects:** Package ML code in a reusable, reproducible form.
3.  **Models:** Manage ML models from various libraries and deploy them to diverse serving platforms.
4.  **Model Registry:** Store, annotate, discover, and manage models in a central repository.

To view the MLflow UI, you would typically run `!mlflow ui` in your terminal or a separate Colab cell if you have a local setup or a tracking server. In Colab, if you want to run `mlflow ui` within the notebook, you would typically need to expose a port, which is a bit more involved. For now, the focus is on logging.

### What is DVC?

DVC (Data Version Control) is an open-source version control system for machine learning projects. It makes ML projects reproducible by tracking machine learning models and datasets. DVC works alongside Git, allowing you to manage large files and directories as part of your Git repository without committing them directly to Git.

Key features include:

1.  **Data Versioning:** Version large datasets and models.
2.  **Experiment Management:** Track experiments by linking code, data, and models.
3.  **Reproducibility:** Recreate experiments by checking out specific data and model versions.
4.  **Pipeline Management:** Define and run data processing and ML pipelines.

In [1]:
!pip install mlflow cryptography==43.0.3 --ignore-installed blinker

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached cryptography-43.0.3-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.14.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached gunicorn-26.0.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached huey

In [2]:
# Install pyngrok to tunnel the MLflow UI
!pip install pyngrok

In [3]:
!pip install dvc[gdrive]

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

In [ ]:
import os
from google.colab import userdata

# Get your ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Set the authtoken
!ngrok authtoken {NGROK_AUTH_TOKEN}

print("ngrok authtoken configured.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Installing DVC

Let's install DVC. We'll use `dvc[gdrive]` for Google Drive integration, which is useful in Colab.

### Basic DVC Usage

To use DVC, you'll typically initialize a Git repository first, and then a DVC repository within it. For simplicity in Colab, we can skip the Git initialization if you only want to version data locally or with Google Drive as a remote. Let's set up a DVC remote to your Google Drive to store versioned data.

**Note:** You need to have git installed and initialize a git repo for DVC to work fully. If you haven't, please run !apt-get update && apt-get install git -y and then !git init in a separate cell before running the DVC commands.

**Setting up DVC with Google Drive:**

### Connecting Colab Git to GitHub

To connect your Colab environment to your GitHub repository, you'll need to configure Git with your user information and then either use an SSH key or a Personal Access Token (PAT) for authentication. Using a PAT is often simpler in Colab.

#### 1. Configure Git User Information

In [ ]:
# Replace with your GitHub email and username
!git config --global user.email "isaac_adebayojr@gmail.com"
!git config --global user.name "isaacadebayo"

print("Git user configured.")

#### 2. Authenticate with GitHub using a Personal Access Token (PAT)

GitHub requires authentication for many Git operations. A Personal Access Token (PAT) is a secure way to do this without exposing your password. If you don't have one, you can create it in your GitHub settings (Settings > Developer settings > Personal access tokens > Tokens (classic)). Give it the `repo` scope.

Once you have your PAT, you can use it to clone your repository.

In [ ]:
# Store your GitHub PAT securely in Colab Secrets
# Click on the '🔑' icon on the left panel, add a new secret named 'GH_TOKEN' and paste your PAT there.
from google.colab import userdata
import os

# Get the PAT from Colab secrets
GH_TOKEN = userdata.get('GH_TOKEN')

# Set it as an environment variable (optional, but good practice if cloning private repos)
os.environ['GH_TOKEN'] = GH_TOKEN

print("GitHub Token retrieved. You can now use it for cloning private repositories.")

#### 3. Clone your GitHub Repository

Now you can clone your repository. For private repositories, you'll include your PAT in the URL.

In [ ]:
# Replace 'your_username' and 'your_repository' with your actual GitHub details.
# For a public repository:
!git clone https://github.com/isaacadebayo/Predictive-Analytics-Public-Datasets.git

# For a private repository using PAT:
# This uses the GH_TOKEN environment variable
!git clone https://{GH_TOKEN}@github.com/isaacadebayo/Predictive-Analytics-Public-Datasets

# Change into your repository directory to perform Git operations
%cd Predictive-Analytics-Public-Datasets

print("Repository cloned. You are now in the repository directory.")

In [ ]:
# Initialize DVC within the current directory (which is the cloned Git repo)
!dvc init

# Configure Google Drive as a remote storage (replace 'dvc_storage' with your desired folder name in Google Drive)
!dvc remote add -d gdrive_remote gdrive://MyDrive/dvc_storage
!dvc remote add -d gcs_remote gs://my-agentic-chatbot-data-lake

# Create a data directory within the Git repository if it doesn't exist
!mkdir -p data

# Copy the data file into the Git repository's data directory
# This ensures the .dvc file (created by dvc add) will be within the Git repo
!cp /content/drive/MyDrive/aapl_stock_prices.csv data/aapl_stock_prices.csv

# Add a data file to DVC. This will create a .dvc file at data/aapl_stock_prices.csv.dvc
!dvc add data/aapl_stock_prices.csv

# To commit the .dvc file to Git (requires Git setup, which is already done)
# Now git will find the .dvc file within its repository's working directory
!git add data/aapl_stock_prices.csv.dvc
!git commit -m "Add aapl_stock_prices.csv to DVC"

print("DVC setup complete. Your data file is now versioned with DVC and stored in Google Drive.")
print("You can push/pull changes using: !dvc push and !dvc pull")

In [ ]:
# Stage all changes in the current directory
!git add .

# Commit the staged changes with a message
# Replace 'Your insightful commit message' with a description of your changes
!git commit -m "Testing commit from colab to github"

# Push the changes to the 'main' branch of your remote repository
# This uses the GH_TOKEN environment variable for authentication
!git push https://{GH_TOKEN}@github.com/isaacadebayo/Predictive-Analytics-Public-Datasets.git main

In [ ]:
print('Contents of the cloned repository:')
!ls -F

After these steps, you can use standard `!git` commands (e.g., `!git add .`, `!git commit -m "Message"`, `!git push origin main`) to interact with your GitHub repository from Colab.

### More on DVC

*   DVC and dataset integrity alteration: DVC focuses on dataset integrity and preventing alteration. Its core purpose is to version control data just like Git version controls code
*   Purpose of .dvc file with cloud storage . dvc does not store the actual data itself instead it is a small human readable text file that contains metadata about data file such as path of originl data file, hash checksum of the data file and a pointer to the locaiton of the actual data in my remote storage





In [ ]:
stocks = pd.read_csv('/content/drive/MyDrive/aapl_stock_prices.csv')
stocks.head()

# Feature Engineering

Moving average price

In [ ]:
stocks['Moving_Average'] = stocks['Close'].rolling(window=50).mean()

stocks

Difference in price

In [ ]:
delta = stocks['Close'].diff()
delta

Separate calculated gain and losses

In [ ]:
gain = delta.clip(lower=0)
loss = -1 * delta.clip(upper=0)

Calculate Exponential moving average

In [ ]:
avg_gain = gain.ewm(com=14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(com=14, min_periods=14, adjust=False).mean()

avg_loss

In [ ]:
rs = avg_gain / avg_loss
rsi = 100 - (100 / (1 + rs))

stocks['RSI'] = rsi
stocks.head()

Drop first 50 rows to incorporate moving average column and prevent skewing the data

In [ ]:
stocks.isna().sum()

In [ ]:
stocks.drop(index=stocks.index[:50], inplace=True)
stocks.head()

In [ ]:
stocks.info()

In [ ]:
if 'Date' in stocks.columns:
    stocks['Date'] = pd.to_datetime(stocks['Date'])
    stocks = stocks.set_index('Date')
stocks.Close.plot(figsize=(18,7), linewidth=1, c='g')

In [ ]:
# Create lagged sequences
window_size = 10

#perm = np.arange(window_size)
perm = np.random.permutation(np.arange(window_size))

X, y = [], []
dates = []
for i in range(len(stocks) - window_size):
    X.append(stocks.iloc[i:i+window_size].values)
    # Predict only the 'close' price (index 1) for y
    y.append(stocks.iloc[i+window_size, 1])
    dates.append(stocks.index[i+window_size])

X = np.array(X)
y = np.array(y)
dates = np.array(dates)

print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# one temporal split, used by everyone
split_idx    = int(len(X) * 0.8)
X_train_3d   = X[:split_idx]       # for LSTM (3D)
X_test_3d    = X[split_idx:]
X_train_2d   = X_train_3d.reshape(X_train_3d.shape[0], -1)  # for sklearn + Dense
X_test_2d    = X_test_3d.reshape(X_test_3d.shape[0], -1)
y_train      = y[:split_idx]
y_test       = y[split_idx:]
dates_train  = dates[:split_idx]
dates_test   = dates[split_idx:]

print(f"Train: X={X_train_3d.shape}, y={y_train.shape}")
print(f"Test:  X={X_test_3d.shape}, y={y_test.shape}")
print(f"Train: X={X_train_2d.shape}, y={y_train.shape}")
print(f"Test:  X={X_test_2d.shape}, y={y_test.shape}")
print(f"Train: X={dates_train.shape}, y={y_train.shape}")
print(f"Test:  X={dates_test.shape}, y={y_test.shape}")


In [ ]:

from sklearn.preprocessing import MinMaxScaler
# one scaler, fit on train only, used by Keras pipelines
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_train_scaled_2d = scaler_X.fit_transform(X_train_2d)
X_test_scaled_2d   = scaler_X.transform(X_test_2d)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1,1))
y_test_scaled  = scaler_y.transform(y_test.reshape(-1,1))
# reshape scaled 2D back to 3D
X_train_scaled_3d = X_train_scaled_2d.reshape(X_train_3d.shape)
X_test_scaled_3d  = X_test_scaled_2d.reshape(X_test_3d.shape)

In [ ]:
plt.figure(figsize=(7,3))
# y_train and y_test are now 1D arrays
plt.plot( dates_train, y_train, 'b', label='Train' );
plt.plot( dates_test, y_test, 'r', label='Test' );
plt.legend()

In [ ]:
'''# Simple index-based split
split_idx = int(len(X) * 0.9)
X_train, y_train, dates_train = X[:split_idx], y[:split_idx], dates[:split_idx]
X_test, y_test, dates_test = X[split_idx:], y[split_idx:], dates[split_idx:]

# Reshape X_train and X_test to be 2D for scikit-learn models
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")'''

# Cross Validation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

models = [
    LinearRegression(),
    KNeighborsRegressor(n_neighbors=4, weights='uniform'),
    RandomForestRegressor(n_estimators=10, max_depth=5),
    RandomForestRegressor(n_estimators=30, max_depth=10),
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2),
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3),
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5),
]

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit

#kf = KFold(n_splits=5, shuffle=True)
kf = TimeSeriesSplit(n_splits=5)

# Reshape X to be 2-dimensional for scikit-learn models
X_2d = X.reshape(X.shape[0], -1)

for m in models:
    print(m)

    cv_scores = -cross_val_score(m, X_2d, y, cv=kf, scoring='neg_mean_absolute_percentage_error')
    cv_scores = pd.DataFrame(cv_scores)
    display(cv_scores.describe().T)

    print()
# end

# Grid SearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Define models and their parameter grids
param_grids = {
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {
            "n_neighbors": [1, 5, 10, 20, 50],
            "weights": ["uniform", "distance"]
        }
    ),

    "RandomForestRegressor": (
        RandomForestRegressor(),
        {
            "n_estimators": [10, 50, 100],
            "max_depth": [3, 5, 10],
            "min_samples_split": [2, 5, 10],
            'min_samples_leaf': [1, 2, 5, 10]
        }
    ),

    "GradientBoostingRegressor": (
        GradientBoostingRegressor(),
        {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 0.2],
            "max_depth": [2, 3, 5],
            'min_samples_leaf': [1, 2, 5, 10]
        }
    )
}

# Define cross-validation strategy
#kf = KFold(n_splits=5, shuffle=True)
kf = TimeSeriesSplit(n_splits=5)

# Store results
results = []

# Perform GridSearchCV for each model
for name, (model, param_grid) in param_grids.items():
    print(f"Performing GridSearchCV for {name}...")

    grid_search = GridSearchCV(model, param_grid, cv=kf, scoring="neg_mean_absolute_percentage_error", n_jobs=-1)
    grid_search.fit(X_2d, y)

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    results.append({"Model": name, "Best Score": best_score, "Best Params": best_params})
# end

results

# Feature Importance










In [ ]:
# Fit a Random Forest model
rf = RandomForestRegressor(n_estimators=1000)
rf.fit(X_2d, y)

# Generate feature names for X_2d
original_features = stocks.columns.tolist()
feature_names_for_X_2d = []
for lag in range(window_size, 0, -1): # Lags from window_size down to 1
    for col in original_features:
        feature_names_for_X_2d.append(f'{col}_lag{lag}')

# Get feature importances
importances = pd.Series(rf.feature_importances_, index=feature_names_for_X_2d).sort_values(ascending=False)
importances = importances[:20]

# Get the names of the selected features
features_rf = importances.index.tolist()

# Create a DataFrame from X_2d with the correct column names for easier slicing
X_2d_df = pd.DataFrame(X_2d, columns=feature_names_for_X_2d)

# Select only the important features from X_2d_df
X_pars = X_2d_df[features_rf]

importances

# KERAS

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, PolynomialDecay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def build_model( X_train, epochs, batch_size, decay_frac, initial_learning_rate, dropout_rate ):

    # Create model
    model = Sequential([
        Input(shape=X_train.shape[1]),
        Dense(20, activation='relu'),
        Dropout(dropout_rate),
        Dense(20, activation='relu'),
        Dropout(dropout_rate),
        Dense(20, activation='relu'),
        Dense(1, activation='relu'),
    ])

    # define training parameters
    decay_epochs    = int( epochs * decay_frac )
    n_train         = X_train.shape[0]
    steps_per_epoch = int( np.ceil( n_train / batch_size ) )
    decay_steps     = decay_epochs * steps_per_epoch

    lr_schedule_poly = PolynomialDecay(
        initial_learning_rate=initial_learning_rate,
        end_learning_rate=0.001,
        decay_steps=decay_steps,
        power=2.0
    )

    # Define the optimizer with a custom learning rate
    optimizer = Adam(
        learning_rate=lr_schedule_poly,
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss='mae'
    )

    return model
# end

In [ ]:
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler # Import MinMaxScaler

# Re-defining build_model with the fix for the Input layer
from keras.models import Sequential
from keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, PolynomialDecay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def build_model( X_train, epochs, batch_size, decay_frac, initial_learning_rate, dropout_rate ):

    # Create model
    model = Sequential([
        Input(shape=(X_train.shape[1], X_train.shape[2])), # Corrected: Input layer now expects 3D shape
        Flatten(), # Added: Flatten the input for Dense layers
        Dense(20, activation='relu'),
        Dropout(dropout_rate),
        Dense(20, activation='relu'),
        Dropout(dropout_rate),
        Dense(20, activation='relu'),
        Dense(1, activation='linear'), # Changed activation to 'linear' for regression
    ])

    # define training parameters
    decay_epochs    = int( epochs * decay_frac )
    n_train         = X_train.shape[0]
    steps_per_epoch = int( np.ceil( n_train / batch_size ) )
    decay_steps     = decay_epochs * steps_per_epoch

    lr_schedule_poly = PolynomialDecay(
        initial_learning_rate=initial_learning_rate,
        end_learning_rate=0.001,
        decay_steps=decay_steps,
        power=2.0
    )

    # Define the optimizer with a custom learning rate
    optimizer = Adam(
        learning_rate=lr_schedule_poly,
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss='mae'
    )

    return model
# end

epochs                = 500
batch_size            = 256
decay_frac            = 0.8
dropout_rate          = 0.25
initial_learning_rate = 0.01

n_splits = 5
#kf = KFold(n_splits=n_splits, shuffle=True)
kf = TimeSeriesSplit(n_splits=n_splits)

train_mae_list = []
test_mae_list = []
train_r2_list = []
test_r2_list = []

fold = 1
for train_index, test_index in kf.split(X):
    print(f"Fold {fold}/{n_splits}")
    X_train_fold, X_test_fold = X[train_index], X[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]

    # Scale X_train_fold and y_train_fold for the neural network
    # Flatten X_train_fold and X_test_fold for scaling, then reshape back
    scaler_X = MinMaxScaler(feature_range=(0,1))
    X_train_flat = X_train_fold.reshape(X_train_fold.shape[0], -1)
    X_test_flat = X_test_fold.reshape(X_test_fold.shape[0], -1)

    X_train_scaled = scaler_X.fit_transform(X_train_flat)
    X_test_scaled = scaler_X.transform(X_test_flat)

    X_train_scaled = X_train_scaled.reshape(X_train_fold.shape)
    X_test_scaled = X_test_scaled.reshape(X_test_fold.shape)

    scaler_y = MinMaxScaler(feature_range=(0,1))
    y_train_scaled = scaler_y.fit_transform(y_train_fold.reshape(-1, 1))
    y_test_scaled = scaler_y.transform(y_test_fold.reshape(-1, 1))

    # Build a fresh model (which resets optimizer state including the learning rate schedule)
    model = build_model(
        X_train_scaled, # Changed to X_train_scaled (3D data for current fold)
        epochs,
        batch_size,
        decay_frac,
        initial_learning_rate,
        dropout_rate
    )

    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor='val_loss',  # Monitor validation loss
        patience=100,          # Stop after 5 epochs without improvement
        restore_best_weights=True  # Restore the best weights after stopping
    )

    # Train the model
    history = model.fit(
        X_train_scaled, y_train_scaled, # Changed to X_train_scaled (3D data for current fold)
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test_scaled, y_test_scaled), # Changed to X_test_scaled (3D data for current fold)
        verbose=0,
        callbacks=[early_stopping]
    )

    # Evaluate on training data
    y_train_pred_scaled = model.predict(X_train_scaled, verbose=0)
    y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled) # Inverse transform predictions
    train_mae = mean_absolute_error(y_train_fold, y_train_pred)
    train_r2 = r2_score(y_train_fold, y_train_pred)

    # Evaluate on test data
    y_test_pred_scaled = model.predict(X_test_scaled, verbose=0)
    y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled) # Inverse transform predictions
    test_mae = mean_absolute_error(y_test_fold, y_test_pred)
    test_r2 = r2_score(y_test_fold, y_test_pred)

    print("MAE:")
    print(f"  Train: {train_mae:.4f}")
    print(f"  Test:  {test_mae:.4f}")
    print("R^2:")
    print(f"  Train: {train_r2:.4f}")
    print(f"  Test:  {test_r2:.4f}")
    print()

    train_mae_list.append(train_mae)
    test_mae_list.append(test_mae)
    train_r2_list.append(train_r2)
    test_r2_list.append(test_r2)

    fold += 1
# end

# Model Creation

In [ ]:
dropout_rate = 0.1

model_dense  = Sequential([
    Input(shape=(X_train_scaled_2d.shape[1],)), # Corrected input shape to match X_train_scaled_2d

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(dropout_rate),

    Dense(16, activation='relu'),
    BatchNormalization(),
    Dropout(dropout_rate),

    Dense(8, activation='relu'),

    Dense(1, activation='linear'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model_dense .compile(
    optimizer=optimizer,
    loss='mae',
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Callback to reduce LR when a monitored metric has stopped improving
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',       # metric to monitor
    factor=0.5,               # factor by which to reduce the LR
    patience=50,              # number of epochs with no improvement after which LR will be reduced
    min_lr=1e-6,              # lower bound on the learning rate
    verbose=1                 # print when LR is reduced
)

model_dense .summary()

### Re-scaling Data for `model_dense`

The cross-validation loop in `cell_KDvNPcdiepgH` updated the `y_train_scaled` variable. To ensure consistent data cardinality for training `model_dense`, we need to re-scale the data (`X_train_2d`, `y_train`, `X_test_2d`, `y_test`) using the original `scaler_X` and `scaler_y` (fitted on the 80/20 split) immediately before training `model_dense`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Re-initialize scalers (or re-use existing ones if already fitted on the correct split)
# scaler_X and scaler_y were defined and fitted in cell 'XlacRcBnxwXz' based on the initial 80/20 split.
# We will re-apply them to ensure consistency.

# Re-scale X_train_2d and X_test_2d
X_train_scaled_2d = scaler_X.transform(X_train_2d)
X_test_scaled_2d  = scaler_X.transform(X_test_2d)

# Re-scale y_train and y_test
y_train_scaled = scaler_y.transform(y_train.reshape(-1,1))
y_test_scaled  = scaler_y.transform(y_test.reshape(-1,1))

print(f"X_train_scaled_2d shape: {X_train_scaled_2d.shape}")
print(f"y_train_scaled shape: {y_train_scaled.shape}")
print(f"X_test_scaled_2d shape: {X_test_scaled_2d.shape}")
print(f"y_test_scaled shape: {y_test_scaled.shape}")

print("Data for model_dense re-scaled successfully.")

In [ ]:
history_dense = model_dense.fit(
    X_train_scaled_2d, y_train_scaled, # Use scaled 2D data
    epochs=150,
    batch_size=64,
    validation_data=(X_test_scaled_2d, y_test_scaled), # Use scaled 2D data for validation
    verbose=1,
    callbacks=[early_stopping]
)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Get predictions on scaled data
y_train_pred_scaled = model_dense.predict(X_train_scaled_2d)[:,0].reshape(-1, 1)
y_test_pred_scaled  = model_dense.predict(X_test_scaled_2d)[:,0].reshape(-1, 1)

# Inverse transform predictions to original scale
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred  = scaler_y.inverse_transform(y_test_pred_scaled).flatten()

# Define true unscaled values for comparison
y_train_true = y_train
y_test_true = y_test

# Create a figure with two subplots side by side
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

# Left subplot: Training and Validation Loss
ax.plot(history_dense.history['loss'], label='Training Loss')
ax.plot(history_dense.history['val_loss'], label='Validation Loss')
ax.set_title('Training and Validation Loss')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()

r2_train = r2_score(y_train_true, y_train_pred)
print(f'r2_train:   {r2_train:.5}')

r2_test = r2_score(y_test_true, y_test_pred)
print(f'r2_test:    {r2_test:.5}')

rmse_train = mean_squared_error(y_train_true, y_train_pred)**0.5
print(f'rmse_train: {rmse_train:.5}')

rmse_test = mean_squared_error(y_test_true, y_test_pred)**0.5
print(f'rmse_test:  {rmse_test:.5}')

y_train_pred = pd.Series(y_train_pred, index=dates_train)
y_test_pred  = pd.Series(y_test_pred, index=dates_test)

# Calculate global y-axis limits
y_min = min(y_train_true.min(), y_train_pred.min(), y_test_true.min(), y_test_pred.min())
y_max = max(y_train_true.max(), y_train_pred.max(), y_test_true.max(), y_test_pred.max())

# Plot train and test side by side with shared y limits
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(dates_train, y_train_true, 'r', label='True')
axs[0].plot(y_train_pred, 'g', label='Predicted')
axs[0].set_title('Train Set')
axs[0].set_ylim(y_min, y_max)
axs[0].legend()

axs[1].plot(dates_test, y_test_true, 'r', label='True')
axs[1].plot(y_test_pred, 'g', label='Predicted')
axs[1].set_title('Test Set')
axs[1].set_ylim(y_min, y_max)
axs[1].legend()

plt.tight_layout()
plt.show()

# Plot train and test side by side with shared y limits
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(y_train_true, y_train_pred.values, 'c.')
axs[0].set_title('Train Set')
# axs[0].legend() # Removed as there's no label for scatter plot

axs[1].plot(y_test_true, y_test_pred.values, 'c.')
axs[1].set_title('Test Set')
# axs[1].legend() # Removed as there's no label for scatter plot

plt.tight_layout()
plt.show()

mae_train = mean_absolute_error(y_train_true, y_train_pred)
r2_train = r2_score(y_train_true, y_train_pred)
mae_test = mean_absolute_error(y_test_true, y_test_pred)
r2_test = r2_score(y_test_true, y_test_pred)

print("MAE:")
print(f"  Train: {mae_train:.4f}")
print(f"  Test:  {mae_test:.4f}")
print("R^2:")
print(f"  Train: {r2_train:.4f}")
print(f"  Test:  {r2_test:.4f}")

In [ ]:
!mlflow ui

### Alternative: Viewing MLflow UI with `localtunnel`

Since `ngrok` requires an authenticated token, we can use `localtunnel` as an alternative. `localtunnel` can also expose your local MLflow UI to a public URL, often without requiring explicit authentication for basic use.


# LSTM

In [ ]:
# Create lagged sequences
window_size = 10

#perm = np.arange(window_size)
perm = np.random.permutation(np.arange(window_size))

X_recent, y_recent = [], []
dates_recent = []
for i in range(len(stocks) - window_size):
    X_recent.append(stocks.iloc[i:i+window_size].values)
    # Predict only the 'close' price (index 1) for y
    y_recent.append(stocks.iloc[i+window_size, 1])
    dates_recent.append(stocks.index[i+window_size])

X_recent = np.array(X_recent)
y_recent = np.array(y_recent)
dates_recent = np.array(dates_recent)

print(f"X shape: {X_recent.shape}, y shape: {y_recent.shape}")

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, PolynomialDecay
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau # new update

# define training parameters
epochs          = 500
batch_size      = 256
decay_epochs    = int( epochs * 0.5 )

n_train         = X_train_3d.shape[0]
steps_per_epoch = int( np.ceil( n_train / batch_size ) )
decay_steps     = decay_epochs * steps_per_epoch
#print(decay_steps)

# Define the learning rate schedule
initial_learning_rate = 0.05

lr_schedule_exp = ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_rate=5.0,
    decay_steps=decay_steps
)

lr_schedule_poly = PolynomialDecay(
    initial_learning_rate=initial_learning_rate,
    end_learning_rate=0.001,
    decay_steps=decay_steps,
    power=1.0
)

# Define the optimizer with a custom learning rate
optimizer = Adam(
    learning_rate=initial_learning_rate,
    #learning_rate=lr_schedule_exp,
    #learning_rate=lr_schedule_poly,
    #clipnorm=1,
    #clipvalue=1,
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=50,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Callback to reduce LR when a monitored metric has stopped improving
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',       # metric to monitor
    factor=0.5,               # factor by which to reduce the LR
    patience=50,              # number of epochs with no improvement after which LR will be reduced
    min_lr=1e-6,              # lower bound on the learning rate
    verbose=1                 # print when LR is reduced
)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='mae'
)

In [ ]:
'''from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=50,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Callback to reduce LR when a monitored metric has stopped improving
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',       # metric to monitor
    factor=0.5,               # factor by which to reduce the LR
    patience=50,              # number of epochs with no improvement after which LR will be reduced
    min_lr=1e-6,              # lower bound on the learning rate
    verbose=1                 # print when LR is reduced
)'''

### Cross validation for LSTM entire stock dataset with scaling

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# Define the LSTM model building function for cross-validation
def build_lstm_model_cv(window_size, num_features, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential([
        Input(shape=(window_size, num_features)),
        LSTM(64, return_sequences=True),
        Dropout(dropout_rate),
        BatchNormalization(),
        LSTM(32, return_sequences=False),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mae')
    return model

# Assuming X_recent and y_recent are already defined and represent the data to be used
# X_recent has shape (n_samples, window_size, n_features)
# y_recent has shape (n_samples,)

# Define cross-validation parameters
n_splits = 5
kf_time_series = TimeSeriesSplit(n_splits=n_splits)

epochs = 150 # adjust this
batch_size = 64 # adjust this

train_mae_cv_list = []
test_mae_cv_list = []
train_r2_cv_list = []
test_r2_cv_list = []

fold = 1
# X_recent and y_recent are 3D and 1D respectively, but TimeSeriesSplit expects 2D array for X
# We will use X_recent_2d and reshape within the loop for LSTM input

# First, reshape X_recent to 2D for TimeSeriesSplit
X_recent_2d = X_recent.reshape(X_recent.shape[0], -1)

for train_index, test_index in kf_time_series.split(X_recent_2d):
    print(f"Fold {fold}/{n_splits}")

    # Split data for the current fold
    X_train_fold_2d, X_test_fold_2d = X_recent_2d[train_index], X_recent_2d[test_index]
    y_train_fold, y_test_fold = y_recent[train_index], y_recent[test_index]

    # Scale X and y within the fold to prevent data leakage
    scaler_X_cv = MinMaxScaler(feature_range=(0, 1))
    scaler_y_cv = MinMaxScaler(feature_range=(0, 1))

    X_train_scaled_2d = scaler_X_cv.fit_transform(X_train_fold_2d)
    X_test_scaled_2d = scaler_X_cv.transform(X_test_fold_2d)

    y_train_scaled = scaler_y_cv.fit_transform(y_train_fold.reshape(-1, 1))
    y_test_scaled = scaler_y_cv.transform(y_test_fold.reshape(-1, 1))

    # Reshape scaled X data to 3D for LSTM input
    current_window_size = X_recent.shape[1] # This should be 10
    current_num_features = X_recent.shape[2] # This should be 5

    X_train_scaled_3d_cv = X_train_scaled_2d.reshape((X_train_scaled_2d.shape[0], current_window_size, current_num_features))
    X_test_scaled_3d_cv = X_test_scaled_2d.reshape((X_test_scaled_2d.shape[0], current_window_size, current_num_features))

    # Build a fresh LSTM model for each fold
    model_lstm_cv = build_lstm_model_cv(current_window_size, current_num_features)

    # Early stopping callback
    early_stopping_cv = EarlyStopping(
        monitor='val_loss',
        patience=25, # Adjusted patience for cross-validation
        restore_best_weights=True
    )

    # Train the model
    history_cv = model_lstm_cv.fit(
        X_train_scaled_3d_cv, y_train_scaled,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test_scaled_3d_cv, y_test_scaled),
        verbose=0,
        callbacks=[early_stopping_cv]
    )

    # Evaluate on training data
    y_train_pred_scaled_cv = model_lstm_cv.predict(X_train_scaled_3d_cv, verbose=0)
    y_train_pred_cv = scaler_y_cv.inverse_transform(y_train_pred_scaled_cv) # Inverse transform predictions
    train_mae_cv = mean_absolute_error(y_train_fold, y_train_pred_cv)
    train_r2_cv = r2_score(y_train_fold, y_train_pred_cv)

    # Evaluate on test data
    y_test_pred_scaled_cv = model_lstm_cv.predict(X_test_scaled_3d_cv, verbose=0)
    y_test_pred_cv = scaler_y_cv.inverse_transform(y_test_pred_scaled_cv) # Inverse transform predictions
    test_mae_cv = mean_absolute_error(y_test_fold, y_test_pred_cv)
    test_r2_cv = r2_score(y_test_fold, y_test_pred_cv)

    print("MAE:")
    print(f"  Train: {train_mae_cv:.4f}")
    print(f"  Test:  {test_mae_cv:.4f}")
    print("R^2:")
    print(f"  Train: {train_r2_cv:.4f}")
    print(f"  Test:  {test_r2_cv:.4f}")
    print()

    train_mae_cv_list.append(train_mae_cv)
    test_mae_cv_list.append(test_mae_cv)
    train_r2_cv_list.append(train_r2_cv)
    test_r2_cv_list.append(test_r2_cv)

    fold += 1
# end

print("\nCross-validation results:")
cv_results_df = pd.DataFrame({
    'Train MAE': train_mae_cv_list,
    'Test MAE': test_mae_cv_list,
    'Train R2': train_r2_cv_list,
    'Test R2': test_r2_cv_list
})
display(cv_results_df)
display(cv_results_df.describe())

In [ ]:
# is this scaled
X_train_2 = X_train_3d # X_train_3d is already in the correct 3D shape (samples, timesteps, features)
X_test_2 = X_test_3d   # X_test_3d is already in the correct 3D shape (samples, timesteps, features)

# Determine the number of features from X_train_3d's last dimension
num_features = X_train_3d.shape[2]

model_lstm = Sequential([
    # The input shape for LSTM layers should be (timesteps, features)
    Input(shape=(window_size, num_features)),

    LSTM(64, return_sequences=True),
    Dropout(0.2),
    BatchNormalization(),

    LSTM(32, return_sequences=False),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(1)
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.001)

# Compile model
model_lstm.compile(
    optimizer=optimizer,
    loss='mae',
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 25 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

model_lstm.summary()

# added this

# Initialize and fit scaler_y_final with the unscaled y_train_fold from the last CV fold
scaler_y_final = MinMaxScaler(feature_range=(0, 1))
scaler_y_final.fit(y_train_fold.reshape(-1, 1))

# after scaling run the lstm model of full stock data
history_lstm = model_lstm.fit(
    X_train_scaled_3d, y_train_scaled, # Use scaled 3D data
    epochs=150,
    batch_size=64,
    validation_data=(X_test_scaled_3d, y_test_scaled), # Use scaled 3D data for validation
    verbose=1,
    callbacks=[early_stopping]
)

In [ ]:
'''history_lstm = model_lstm.fit(
    X_train_scaled_3d, y_train_scaled, # Use scaled 3D data
    epochs=150,
    batch_size=64,
    validation_data=(X_test_scaled_3d, y_test_scaled), # Use scaled 3D data for validation
    verbose=1,
    callbacks=[early_stopping]
)'''

In [ ]:
# Get predictions on scaled data
y_train_pred_scaled = model_lstm.predict(X_train_scaled_3d)[:,0].reshape(-1, 1)
y_test_pred_scaled  = model_lstm.predict(X_test_scaled_3d)[:,0].reshape(-1, 1)

# Inverse transform predictions to original scale
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred  = scaler_y.inverse_transform(y_test_pred_scaled).flatten()

# Now use the true values and dates from the train_test_split
y_train_true = y_train
y_test_true = y_test
dates_train_plot = dates_train
dates_test_plot = dates_test

# Create a figure with one subplot for loss
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

# Plot Training and Validation Loss
ax.plot(history_lstm.history['loss'], label='Training Loss')
ax.plot(history_lstm.history['val_loss'], label='Validation Loss')
ax.set_title('Training and Validation Loss (LSTM)')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

# Calculate metrics for LSTM model
r2_train = r2_score(y_train_true, y_train_pred)
print(f'r2_train:   {r2_train:.5}')

r2_test = r2_score(y_test_true, y_test_pred)
print(f'r2_test:    {r2_test:.5}')

rmse_train = mean_squared_error(y_train_true, y_train_pred)**0.5
print(f'rmse_train: {rmse_train:.5}')

rmse_test = mean_squared_error(y_test_true, y_test_pred)**0.5
print(f'rmse_test:  {rmse_test:.5}')

# Convert predictions to pandas Series with dates as index for plotting
y_train_pred_series = pd.Series(y_train_pred.flatten(), index=dates_train_plot)
y_test_pred_series  = pd.Series(y_test_pred.flatten(), index=dates_test_plot)

# Calculate global y-axis limits for consistent plotting
y_min = min(y_train_true.min(), y_train_pred_series.min(), y_test_true.min(), y_test_pred_series.min())
y_max = max(y_train_true.max(), y_train_pred_series.max(), y_test_true.max(), y_test_pred_series.max())

# Plot train and test side by side with shared y limits
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(dates_train_plot, y_train_true, 'r', label='True')
axs[0].plot(y_train_pred_series, 'g', label='Predicted')
axs[0].set_title('Train Set (LSTM)')
axs[0].set_ylim(y_min, y_max)
axs[0].legend()

axs[1].plot(dates_test_plot, y_test_true, 'r', label='True')
axs[1].plot(y_test_pred_series, 'g', label='Predicted')
axs[1].set_title('Test Set (LSTM)')
axs[1].set_ylim(y_min, y_max)
axs[1].legend()

plt.tight_layout()
plt.show()

# Plot scatter plots for train and test side by side with shared y limits
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(y_train_true, y_train_pred_series.values, 'c.')
axs[0].set_title('Train Set (LSTM)')
axs[0].set_xlabel('True Values')
axs[0].set_ylabel('Predicted Values')
# axs[0].legend() # Removed as there's no label for scatter plot

axs[1].plot(y_test_true, y_test_pred_series.values, 'c.')
axs[1].set_title('Test Set (LSTM)')
axs[1].set_xlabel('True Values')
axs[1].set_ylabel('Predicted Values')
# axs[1].legend() # Removed as there's no label for scatter plot

plt.tight_layout()
plt.show()

mae_train = mean_absolute_error(y_train_true, y_train_pred)
mae_test = mean_absolute_error(y_test_true, y_test_pred)

print("MAE:")
print(f"  Train: {mae_train:.4f}")
print(f"  Test:  {mae_test:.4f}")
print("R^2:")
print(f"  Train: {r2_train:.4f}")
print(f"  Test:  {r2_test:.4f}")

#Gemini Suggestion and my fix

### Scaling Data for LSTM Training

The previous LSTM model was trained on unscaled data. It's crucial for neural networks like LSTMs to train on scaled data for better convergence and performance. We will now apply `MinMaxScaler` to the training and testing sets of `X` and `y` that were derived from the feature-selected data (`X_pars`).

The `scaler_X_final` will be fitted on the 2D `X_train` (before reshaping to 3D for LSTM) and `scaler_y_final` will be fitted on `y_train`. These scalers will then be used to transform both the training and testing sets.

*********************************************
### If it doesnt work run lstm in new notebook
*********************************************

In [ ]:
recent_stocks = stocks.tail(574)
recent_stocks

In [ ]:
# Create lagged sequences
window_size = 10

#perm = np.arange(window_size)
perm = np.random.permutation(np.arange(window_size))

X_recent, y_recent = [], []
dates_recent = []
for i in range(len(recent_stocks) - window_size):
    X_recent.append(recent_stocks.iloc[i:i+window_size].values)
    # Predict only the 'close' price (index 1) for y
    y_recent.append(recent_stocks.iloc[i+window_size, 1])
    dates_recent.append(recent_stocks.index[i+window_size])

X_recent = np.array(X_recent)
y_recent = np.array(y_recent)
dates_recent = np.array(dates_recent)

print(f"X shape: {X_recent.shape}, y shape: {y_recent.shape}")

### Time Series Cross-Validation for LSTM Model Scaled Data

This provide a more robust evaluation of the model's performance across different time periods, similar to how other models were evaluated.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# Define the LSTM model building function for cross-validation
def build_lstm_model_cv(window_size, num_features, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential([
        Input(shape=(window_size, num_features)),
        LSTM(64, return_sequences=True),
        Dropout(dropout_rate),
        BatchNormalization(),
        LSTM(32, return_sequences=False),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mae')
    return model

# Assuming X_recent and y_recent are already defined and represent the data to be used
# X_recent has shape (n_samples, window_size, n_features)
# y_recent has shape (n_samples,)

# Define cross-validation parameters
n_splits = 5
kf_time_series = TimeSeriesSplit(n_splits=n_splits)

epochs = 150 # adjust this
batch_size = 64 # adjust this

train_mae_cv_list = []
test_mae_cv_list = []
train_r2_cv_list = []
test_r2_cv_list = []

fold = 1
# X_recent and y_recent are 3D and 1D respectively, but TimeSeriesSplit expects 2D array for X
# We will use X_recent_2d and reshape within the loop for LSTM input

# First, reshape X_recent to 2D for TimeSeriesSplit
X_recent_2d = X_recent.reshape(X_recent.shape[0], -1)

for train_index, test_index in kf_time_series.split(X_recent_2d):
    print(f"Fold {fold}/{n_splits}")

    # Split data for the current fold
    X_train_fold_2d, X_test_fold_2d = X_recent_2d[train_index], X_recent_2d[test_index]
    y_train_fold, y_test_fold = y_recent[train_index], y_recent[test_index]

    # Scale X and y within the fold to prevent data leakage
    scaler_X_cv = MinMaxScaler(feature_range=(0, 1))
    scaler_y_cv = MinMaxScaler(feature_range=(0, 1))

    X_train_scaled_2d = scaler_X_cv.fit_transform(X_train_fold_2d)
    X_test_scaled_2d = scaler_X_cv.transform(X_test_fold_2d)

    y_train_scaled = scaler_y_cv.fit_transform(y_train_fold.reshape(-1, 1))
    y_test_scaled = scaler_y_cv.transform(y_test_fold.reshape(-1, 1))

    # Reshape scaled X data to 3D for LSTM input
    current_window_size = X_recent.shape[1] # This should be 10
    current_num_features = X_recent.shape[2] # This should be 5

    X_train_scaled_3d_cv = X_train_scaled_2d.reshape((X_train_scaled_2d.shape[0], current_window_size, current_num_features))
    X_test_scaled_3d_cv = X_test_scaled_2d.reshape((X_test_scaled_2d.shape[0], current_window_size, current_num_features))

    # Build a fresh LSTM model for each fold
    model_lstm_cv = build_lstm_model_cv(current_window_size, current_num_features)

    # Early stopping callback
    early_stopping_cv = EarlyStopping(
        monitor='val_loss',
        patience=25, # Adjusted patience for cross-validation
        restore_best_weights=True
    )

    # Train the model
    history_cv = model_lstm_cv.fit(
        X_train_scaled_3d_cv, y_train_scaled,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test_scaled_3d_cv, y_test_scaled),
        verbose=0,
        callbacks=[early_stopping_cv]
    )

    # Evaluate on training data
    y_train_pred_scaled_cv = model_lstm_cv.predict(X_train_scaled_3d_cv, verbose=0)
    y_train_pred_cv = scaler_y_cv.inverse_transform(y_train_pred_scaled_cv) # Inverse transform predictions
    train_mae_cv = mean_absolute_error(y_train_fold, y_train_pred_cv)
    train_r2_cv = r2_score(y_train_fold, y_train_pred_cv)

    # Evaluate on test data
    y_test_pred_scaled_cv = model_lstm_cv.predict(X_test_scaled_3d_cv, verbose=0)
    y_test_pred_cv = scaler_y_cv.inverse_transform(y_test_pred_scaled_cv) # Inverse transform predictions
    test_mae_cv = mean_absolute_error(y_test_fold, y_test_pred_cv)
    test_r2_cv = r2_score(y_test_fold, y_test_pred_cv)

    print("MAE:")
    print(f"  Train: {train_mae_cv:.4f}")
    print(f"  Test:  {test_mae_cv:.4f}")
    print("R^2:")
    print(f"  Train: {train_r2_cv:.4f}")
    print(f"  Test:  {test_r2_cv:.4f}")
    print()

    train_mae_cv_list.append(train_mae_cv)
    test_mae_cv_list.append(test_mae_cv)
    train_r2_cv_list.append(train_r2_cv)
    test_r2_cv_list.append(test_r2_cv)

    fold += 1
# end

print("\nCross-validation results:")
cv_results_df = pd.DataFrame({
    'Train MAE': train_mae_cv_list,
    'Test MAE': test_mae_cv_list,
    'Train R2': train_r2_cv_list,
    'Test R2': test_r2_cv_list
})
display(cv_results_df)
display(cv_results_df.describe())

### Defining training parameter

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, PolynomialDecay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input

# define training parameters
epochs          = 250
batch_size      = 256
decay_epochs    = int( epochs * 0.5 )

n_train         = X_train_3d.shape[0]
steps_per_epoch = int( np.ceil( n_train / batch_size ) )
decay_steps     = decay_epochs * steps_per_epoch
#print(decay_steps)

# Define the learning rate schedule
initial_learning_rate = 0.001 # Changed from 0.05 to 0.001 to match CV setup

lr_schedule_exp = ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_rate=5.0,
    decay_steps=decay_steps
)

lr_schedule_poly = PolynomialDecay(
    initial_learning_rate=initial_learning_rate,
    end_learning_rate=0.001,
    decay_steps=decay_steps,
    power=1.0
)

# Define model_lstm_final (moved from a later cell)
window_size = 10 # Assuming this is consistent
num_features = X_train_3d.shape[2] # Corrected: Derive num_features from the actual number of features

model_lstm_final = Sequential([
    Input(shape=(window_size, num_features)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    BatchNormalization(),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

# Define the optimizer with a custom learning rate
optimizer = Adam(
    learning_rate=initial_learning_rate,
    #learning_rate=lr_schedule_exp,
    #learning_rate=lr_schedule_poly,
    #clipnorm=1,
    #clipvalue=1,
)

# Compile model
model_lstm_final.compile(
    optimizer=optimizer,
    loss='mae'
)

### Retraining LSTM Model with Scaled Data

Now we will define and retrain the LSTM model using the newly scaled `X_train_scaled_3d` and `y_train_scaled` data. We will use the same architecture and early stopping mechanism as before. After training, we will evaluate its performance.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler # Import MinMaxScaler

# Correctly determine the number of features from the scaled 3D data
num_features = X_train_scaled_3d_cv.shape[2]

# Define the LSTM model with the same architecture as before
model_lstm_final = Sequential([
    Input(shape=(window_size, num_features)), # Use the correctly calculated num_features

    LSTM(64, return_sequences=True),
    Dropout(0.2),
    BatchNormalization(),

    LSTM(32, return_sequences=False),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(1)
])

# Define the optimizer and compile the model
optimizer = Adam(learning_rate=0.001) # Use the same learning rate as previous LSTM
model_lstm_final.compile(
    optimizer=optimizer,
    loss='mae',
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 25 epochs without improvement (consistent with previous)
    restore_best_weights=True  # Restore the best weights after stopping
)

model_lstm_final.summary()

# --- Start of added check for data cardinality ---
#print(f"Checking X_train_scaled_3d shape: {X_train_scaled_3d_cv.shape}")
#print(f"Checking y_train_scaled shape: {y_train_scaled.shape}")

#if X_train_scaled_3d.shape[0] != y_train_scaled.shape[0]:
#    raise ValueError(f"Data cardinality mismatch: X_train has {X_train_scaled_3d.shape[0]} samples, but y_train has {y_train_scaled.shape[0]} samples.")
# --- End of added check for data cardinality ---

# Initialize and fit scaler_y_final with the unscaled y_train_fold from the last CV fold
scaler_y_final = MinMaxScaler(feature_range=(0, 1))
scaler_y_final.fit(y_train_fold.reshape(-1, 1))

# Train the model with scaled data
history_lstm_final = model_lstm_final.fit(
    X_train_scaled_3d_cv, y_train_scaled,
    epochs=250, # Use same number of epochs
    batch_size=64, # Reverting batch size as per user's preference
    validation_data=(X_test_scaled_3d_cv, y_test_scaled),
    verbose=1,
    callbacks=[early_stopping]
)

# Get predictions using the final model
y_train_pred_scaled = model_lstm_final.predict(X_train_scaled_3d_cv)
y_test_pred_scaled  = model_lstm_final.predict(X_test_scaled_3d_cv)

# Inverse transform predictions to original scale
y_train_pred = scaler_y_final.inverse_transform(y_train_pred_scaled)
y_test_pred  = scaler_y_final.inverse_transform(y_test_pred_scaled)

# Inverse transform true values for consistent comparison based on the cv fold
y_train_true_original = y_train_fold
y_test_true_original = y_test_fold

# Calculate metrics for the retrained LSTM model
r2_train = r2_score(y_train_true_original, y_train_pred)
print(f'r2_train:   {r2_train:.5}')

r2_test = r2_score(y_test_true_original, y_test_pred)
print(f'r2_test:    {r2_test:.5}')

rmse_train = mean_squared_error(y_train_true_original, y_train_pred)**0.5
print(f'rmse_train: {rmse_train:.5}')

rmse_test = mean_squared_error(y_test_true_original, y_test_pred)**0.5
print(f'rmse_test:  {rmse_test:.5}')

In [ ]:
n_splits = 5 # Assuming n_splits is 5 as used in previous CV
kf_time_series_reget = TimeSeriesSplit(n_splits=n_splits)

last_train_index = None
last_test_index = None
last_y_train_fold = None
last_y_test_fold = None
last_scaler_y_cv = None

# Iterate through the splits to get the data and scaler for the last fold
# X_recent_2d is assumed to be available from prior cells.
for i, (tr_idx, ts_idx) in enumerate(kf_time_series_reget.split(X_recent_2d)):
    if i == n_splits - 1: # This is the last fold which is fold 5
        last_train_index = tr_idx
        last_test_index = ts_idx
        last_y_train_fold = y_recent[tr_idx]
        last_y_test_fold = y_recent[ts_idx]

        # Recreate scaler_y_cv for the last fold to ensure consistency
        temp_scaler_y_cv = MinMaxScaler(feature_range=(0,1))
        temp_scaler_y_cv.fit(last_y_train_fold.reshape(-1, 1))
        last_scaler_y_cv = temp_scaler_y_cv
        break


y_train_pred_scaled = model_lstm_final.predict(X_train_scaled_3d_cv)
y_test_pred_scaled  = model_lstm_final.predict(X_test_scaled_3d_cv)

# Inverse transform predictions to original scale using the correct scaler_y_cv
y_train_pred = last_scaler_y_cv.inverse_transform(y_train_pred_scaled)
y_test_pred  = last_scaler_y_cv.inverse_transform(y_test_pred_scaled)

# Now use the true values and dates from the last fold's split
y_train_true = last_y_train_fold.reshape(-1, 1)
y_test_true = last_y_test_fold.reshape(-1, 1)
dates_train_plot = dates_recent[last_train_index]
dates_test_plot = dates_recent[last_test_index]

# Create a figure with one subplot for loss
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

# Plot Training and Validation Loss
ax.plot(history_lstm_final.history['loss'], label='Training Loss')
ax.plot(history_lstm_final.history['val_loss'], label='Validation Loss')
ax.set_title('Training and Validation Loss (LSTM)')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

# Calculate metrics for LSTM model
r2_train = r2_score(y_train_true, y_train_pred)
print(f'r2_train:   {r2_train:.5}')

r2_test = r2_score(y_test_true, y_test_pred)
print(f'r2_test:    {r2_test:.5}')

rmse_train = mean_squared_error(y_train_true, y_train_pred)**0.5
print(f'rmse_train: {rmse_train:.5}')

rmse_test = mean_squared_error(y_test_true, y_test_pred)**0.5
print(f'rmse_test:  {rmse_test:.5}')

# Convert predictions to pandas Series with dates as index for plotting
y_train_pred_series = pd.Series(y_train_pred.flatten(), index=dates_train_plot)
y_test_pred_series  = pd.Series(y_test_pred.flatten(), index=dates_test_plot)

# Calculate global y-axis limits for consistent plotting
y_min = min(y_train_true.min(), y_train_pred_series.min(), y_test_true.min(), y_test_pred_series.min())
y_max = max(y_train_true.max(), y_train_pred_series.max(), y_test_true.max(), y_test_pred_series.max())

# Plot train and test side by side with shared y limits
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(dates_train_plot, y_train_true, 'r', label='True')
axs[0].plot(y_train_pred_series, 'g', label='Predicted')
axs[0].set_title('Train Set (LSTM)')
axs[0].set_ylim(y_min, y_max)
axs[0].legend()

axs[1].plot(dates_test_plot, y_test_true, 'r', label='True')
axs[1].plot(y_test_pred_series, 'g', label='Predicted')
axs[1].set_title('Test Set (LSTM)')
axs[1].set_ylim(y_min, y_max)
axs[1].legend()

plt.tight_layout()
plt.show()

# Plot scatter plots for train and test side by side with shared y limits
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(y_train_true, y_train_pred_series.values, 'c.')
axs[0].set_title('Train Set (LSTM)')
axs[0].set_xlabel('True Values')
axs[0].set_ylabel('Predicted Values')
# axs[0].legend() # Removed as there's no label for scatter plot

axs[1].plot(y_test_true, y_test_pred_series.values, 'c.')
axs[1].set_title('Test Set (LSTM)')
axs[1].set_xlabel('True Values')
axs[1].set_ylabel('Predicted Values')
# axs[1].legend() # Removed as there's no label for scatter plot

plt.tight_layout()
plt.show()

mae_train = mean_absolute_error(y_train_true, y_train_pred)
mae_test = mean_absolute_error(y_test_true, y_test_pred)

print("MAE:")
print(f"  Train: {mae_train:.4f}")
print(f"  Test:  {mae_test:.4f}")
print("R^2:")
print(f"  Train: {r2_train:.4f}")
print(f"  Test:  {r2_test:.4f}")

In [ ]:
# Create a DataFrame to organize the data of true vs pred data graph
predicted_results_df_corrected = pd.DataFrame({
    'Actual Close Price': pd.Series(y_test_true.flatten(), index=dates_test_plot), # Use y_test_true and dates_test_plot
    'Predicted Stock Price': pd.Series(y_test_pred.flatten(), index=dates_test_plot),
    'Difference (Error)': (pd.Series(y_test_true.flatten(), index=dates_test_plot) - pd.Series(y_test_pred.flatten(), index=dates_test_plot)),
    'Percentage Error (%)': ((abs(pd.Series(y_test_true.flatten(), index=dates_test_plot) - pd.Series(y_test_pred.flatten(), index=dates_test_plot))) / pd.Series(y_test_true.flatten(), index=dates_test_plot)) * 100
})

# Get the original 'Open' prices from the stocks DataFrame using dates_test_plot's index
# Note: 'stocks' DataFrame has 'Open' (capital O) column
open_prices = stocks.loc[dates_test_plot, ['Open']]

# Merge this information into predicted_results_df, aligning by index
predicted_results_df_corrected = predicted_results_df_corrected.merge(open_prices, left_index=True, right_index=True)

# Reorder columns for clarity, putting 'Open' and 'Actual Close Price' first
predicted_results_df_corrected = predicted_results_df_corrected[[
    'Open', 'Actual Close Price', 'Predicted Stock Price',
    'Difference (Error)', 'Percentage Error (%)'
]]

# Sort the DataFrame by its index (dates) to enable proper time-series slicing
predicted_results_df_corrected = predicted_results_df_corrected.sort_index()

# Display the corrected numerical table filtered from 2007 to 2026
display(predicted_results_df_corrected.loc['2025':'2026'])

### Saving the Trained Model and Scalers
Save the retrained LSTM model (`lstm_model.keras`), the fitted X-scaler (`scaler_X.pkl`), and the fitted y-scaler (`scaler_y.pkl`) to project directory. These files can be loaded later to make new predictions without retraining or re-fitting the scalers. This uses the last fold (fold 5) of the cross validation

In [ ]:
'''# Save the LSTM model
model_lstm_final.save('lstm_model.keras')
print("LSTM model saved as 'lstm_model.keras'")

# Save the X scaler
joblib.dump(scaler_X_final, 'scaler_X.pkl')
print("X scaler saved as 'scaler_X.pkl'")

# Save the y scaler
joblib.dump(scaler_y_final, 'scaler_y.pkl')
print("Y scaler saved as 'scaler_y.pkl'")'''

### Training model_lstm_final on a Specific Fold (e.g., Fold 4)

To demonstrate how to train and evaluate the model_lstm_final on a specific fold, we will perform the following steps for **Fold 4**:

1.  **Iterate through the TimeSeriesSplit** to get the train_index and test_index for Fold 4.
2.  **Extract the data** (X_train_fold_4, y_train_fold_4, X_test_fold_4, y_test_fold_4, dates_train_fold_4, dates_test_fold_4).
3.  **Scale the data** using new MinMaxScaler instances (scaler_X_fold_4, scaler_y_fold_4) to prevent data leakage from other folds.
4.  **Reshape the scaled X data** to the 3D format required by the LSTM model.
5.  **Build a fresh model_lstm_final** with its optimizer and compile it.
6.  **Train the model** using the specific fold's scaled data.
7.  **Evaluate and plot the results** for Fold 4.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Define the LSTM model building function
def build_lstm_model(window_size, num_features, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential([
        Input(shape=(window_size, num_features)),
        LSTM(64, return_sequences=True),
        Dropout(dropout_rate),
        BatchNormalization(),
        LSTM(32, return_sequences=False),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mae')
    return model

# Re-initialize TimeSeriesSplit
n_splits = 5
kf_time_series = TimeSeriesSplit(n_splits=n_splits)

# X_recent_2d was already created in cell 5fe8341e
# We will iterate to get the 4th fold's indices

selected_fold = 4
current_fold = 0

for train_index, test_index in kf_time_series.split(X_recent_2d):
    current_fold += 1
    if current_fold == selected_fold:
        # Store the indices for the selected fold
        train_index_4 = train_index
        test_index_4 = test_index
        break

print(f"Extracted indices for Fold {selected_fold}")

# Extract data for Fold 4
X_train_fold_4_2d = X_recent_2d[train_index_4]
X_test_fold_4_2d  = X_recent_2d[test_index_4]
y_train_fold_4    = y_recent[train_index_4]
y_test_fold_4     = y_recent[test_index_4]
dates_train_fold_4 = dates_recent[train_index_4]
dates_test_fold_4  = dates_recent[test_index_4]

# Scale data for Fold 4
scaler_X_fold_4 = MinMaxScaler(feature_range=(0, 1))
scaler_y_fold_4 = MinMaxScaler(feature_range=(0, 1))

X_train_scaled_fold_4_2d = scaler_X_fold_4.fit_transform(X_train_fold_4_2d)
X_test_scaled_fold_4_2d  = scaler_X_fold_4.transform(X_test_fold_4_2d)

y_train_scaled_fold_4 = scaler_y_fold_4.fit_transform(y_train_fold_4.reshape(-1, 1))
y_test_scaled_fold_4  = scaler_y_fold_4.transform(y_test_fold_4.reshape(-1, 1))

# Reshape scaled X data to 3D for LSTM input
current_window_size = X_recent.shape[1] # This should be 10
current_num_features = X_recent.shape[2] # This should be 5

X_train_scaled_3d_fold_4 = X_train_scaled_fold_4_2d.reshape((X_train_scaled_fold_4_2d.shape[0], current_window_size, current_num_features))
X_test_scaled_3d_fold_4  = X_test_scaled_fold_4_2d.reshape((X_test_scaled_fold_4_2d.shape[0], current_window_size, current_num_features))

print("Data for Fold 4 prepared and scaled.")

# Build a fresh model for Fold 4
model_lstm_fold_4 = build_lstm_model(
    current_window_size,
    current_num_features,
    dropout_rate=0.2,
    learning_rate=0.001 # Use the optimized learning rate
)

# Early stopping callback
early_stopping_fold_4 = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

print(f"\nTraining model for Fold {selected_fold}...")
history_lstm_fold_4 = model_lstm_fold_4.fit(
    X_train_scaled_3d_fold_4, y_train_scaled_fold_4,
    epochs=250,
    batch_size=64,
    validation_data=(X_test_scaled_3d_fold_4, y_test_scaled_fold_4),
    verbose=1,
    callbacks=[early_stopping_fold_4]
)

print(f"\nEvaluation for Fold {selected_fold}:")
# Get predictions
y_train_pred_scaled_fold_4 = model_lstm_fold_4.predict(X_train_scaled_3d_fold_4)
y_test_pred_scaled_fold_4  = model_lstm_fold_4.predict(X_test_scaled_3d_fold_4)

# Inverse transform predictions to original scale
y_train_pred_fold_4 = scaler_y_fold_4.inverse_transform(y_train_pred_scaled_fold_4)
y_test_pred_fold_4  = scaler_y_fold_4.inverse_transform(y_test_pred_scaled_fold_4)

# Calculate metrics
r2_train_fold_4 = r2_score(y_train_fold_4, y_train_pred_fold_4)
r2_test_fold_4  = r2_score(y_test_fold_4, y_test_pred_fold_4)

rmse_train_fold_4 = mean_squared_error(y_train_fold_4, y_train_pred_fold_4)**0.5
rmse_test_fold_4  = mean_squared_error(y_test_fold_4, y_test_pred_fold_4)**0.5

mae_train_fold_4 = mean_absolute_error(y_train_fold_4, y_train_pred_fold_4)
mae_test_fold_4  = mean_absolute_error(y_test_fold_4, y_test_pred_fold_4)

print("MAE:")
print(f"  Train: {mae_train_fold_4:.4f}")
print(f"  Test:  {mae_test_fold_4:.4f}")
print("R^2:")
print(f"  Train: {r2_train_fold_4:.4f}")
print(f"  Test:  {r2_test_fold_4:.4f}")

# Plotting for Fold 4
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(history_lstm_fold_4.history['loss'], label='Training Loss')
ax.plot(history_lstm_fold_4.history['val_loss'], label='Validation Loss')
ax.set_title(f'Training and Validation Loss (LSTM) - Fold {selected_fold}')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

y_train_pred_series_fold_4 = pd.Series(y_train_pred_fold_4.flatten(), index=dates_train_fold_4)
y_test_pred_series_fold_4  = pd.Series(y_test_pred_fold_4.flatten(), index=dates_test_fold_4)

y_min_fold_4 = min(y_train_fold_4.min(), y_train_pred_series_fold_4.min(), y_test_fold_4.min(), y_test_pred_series_fold_4.min())
y_max_fold_4 = max(y_train_fold_4.max(), y_train_pred_series_fold_4.max(), y_test_fold_4.max(), y_test_pred_series_fold_4.max())

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(dates_train_fold_4, y_train_fold_4, 'r', label='True')
axs[0].plot(y_train_pred_series_fold_4, 'g', label='Predicted')
axs[0].set_title(f'Train Set (LSTM) - Fold {selected_fold}')
axs[0].set_ylim(y_min_fold_4, y_max_fold_4)
axs[0].legend()

axs[1].plot(dates_test_fold_4, y_test_fold_4, 'r', label='True')
axs[1].plot(y_test_pred_series_fold_4, 'g', label='Predicted')
axs[1].set_title(f'Test Set (LSTM) - Fold {selected_fold}')
axs[1].set_ylim(y_min_fold_4, y_max_fold_4)
axs[1].legend()

plt.tight_layout()
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(y_train_fold_4, y_train_pred_series_fold_4.values, 'c.')
axs[0].set_title(f'Train Set (LSTM) - Fold {selected_fold}')
axs[0].set_xlabel('True Values')
axs[0].set_ylabel('Predicted Values')

axs[1].plot(y_test_fold_4, y_test_pred_series_fold_4.values, 'c.')
axs[1].set_title(f'Test Set (LSTM) - Fold {selected_fold}')
axs[1].set_xlabel('True Values')
axs[1].set_ylabel('Predicted Values')

plt.tight_layout()
plt.show()

### Saving the Trained Model for Specific Fold (e.g., Fold 4)

Now, we will save the `model_lstm_fold_4` which was specifically trained and evaluated on **Fold 4**. This allows you to retain the model's weights and configuration from this particular cross-validation split.

In [ ]:
import joblib
model_lstm_fold_4.save('lstm_model_fold_4.keras')
print("LSTM model for Fold 4 saved as 'lstm_model_fold_4.keras'")

# Save the X scaler specific to Fold 4
joblib.dump(scaler_X_fold_4, 'scaler_X_fold_4.pkl')
print("X scaler for Fold 4 saved as 'scaler_X_fold_4.pkl'")

# Save the y scaler specific to Fold 4
joblib.dump(scaler_y_fold_4, 'scaler_y_fold_4.pkl')
print("Y scaler for Fold 4 saved as 'scaler_y_fold_4.pkl'")

### MLflow Tracking Example for LSTM Model

Logging LSTM model's training details, parameters, and performance metrics to MLflow. The results from the `model_lstm_fold_4` which was trained on Fold 4 of the time series cross-validation.

In [ ]:
import mlflow
import mlflow.keras
from sklearn.metrics import mean_absolute_error, r2_score

# Set an experiment name for the LSTM model
mlflow.set_experiment("Stock Price Prediction - LSTM Model")

with mlflow.start_run():
    # Log parameters specific to the LSTM model (from build_lstm_model and training for Fold 4)
    mlflow.log_param("epochs", 250)
    mlflow.log_param("batch_size", 64)
    mlflow.log_param("dropout_rate", 0.2)
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("loss_function", "mae")
    mlflow.log_param("window_size", 10)
    mlflow.log_param("num_features", X_recent.shape[2]) # Get from X_recent as used in CV

    # Log metrics (using the results from the Fold 4 evaluation)
    mlflow.log_metric("train_mae", mae_train_fold_4)
    mlflow.log_metric("test_mae", mae_test_fold_4)
    mlflow.log_metric("train_r2", r2_train_fold_4)
    mlflow.log_metric("test_r2", r2_test_fold_4)
    mlflow.log_metric("train_rmse", rmse_train_fold_4)
    mlflow.log_metric("test_rmse", rmse_test_fold_4)

    # Log the Keras LSTM model for Fold 4
    mlflow.keras.log_model(model_lstm_fold_4, "lstm_model_fold_4")

    print("MLflow Run for LSTM model completed.")

### mlflow for keras

In [ ]:
!ps aux | grep mlflow

In [ ]:
print('Checking active processes on port 5000:')
!lsof -i :5000
# If lsof is not installed, you can try:
#!netstat -tuln | grep 5000
#!fuser 5000/tcp

In [ ]:
!ngrok version

In [ ]:
'''import mlflow
import mlflow.keras
from sklearn.metrics import mean_absolute_error, r2_score

# Set an experiment name
mlflow.set_experiment("Stock Price Prediction - Dense Model")

with mlflow.start_run():
    # Log parameters (you can log any hyperparameter or configuration here)
    mlflow.log_param("epochs", 150)
    mlflow.log_param("batch_size", 64)
    mlflow.log_param("dropout_rate", 0.1)
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("loss_function", "mae")

    # Log metrics
    mlflow.log_metric("train_mae", mae_train)
    mlflow.log_metric("test_mae", mae_test)
    mlflow.log_metric("train_r2", r2_train)
    mlflow.log_metric("test_r2", r2_test)

    # Log the Keras model
    mlflow.keras.log_model(model_dense, "dense_model")

    print("MLflow Run completed. View with: !mlflow ui")'''

Using cloudfare for MLflow, https and avoid the http of ngronk using .app

In [ ]:
import subprocess, time, re, requests

# ── 1. Kill everything ────────────────────────────────────────────
subprocess.run("pkill -f mlflow", shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
subprocess.run("pkill -f proxy.py", shell=True)
time.sleep(4)

# ── 2. Start MLflow ───────────────────────────────────────────────
subprocess.Popen(
    "mlflow ui --host 0.0.0.0 --port 5000 > mlflow_ui.log 2>&1",
    shell=True
)
print("Waiting for MLflow...")
time.sleep(10)

# ── 3. Write proxy ────────────────────────────────────────────────
proxy_code = """
import http.server, urllib.request, urllib.error

class Proxy(http.server.BaseHTTPRequestHandler):
    def _proxy(self, body=None):
        url = f"http://localhost:5000{self.path}"
        headers = {k: v for k, v in self.headers.items()}
        headers["Host"] = "localhost:5000"
        req = urllib.request.Request(url, headers=headers, method=self.command, data=body)
        try:
            resp = urllib.request.urlopen(req)
            self.send_response(resp.status)
            for k, v in resp.headers.items():
                self.send_header(k, v)
            self.end_headers()
            self.wfile.write(resp.read())
        except urllib.error.HTTPError as e:
            self.send_response(e.code)
            self.end_headers()
            self.wfile.write(e.read())
        except Exception as e:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(e).encode())

    def do_GET(self):    self._proxy()
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length))
    def do_PUT(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length))
    def do_DELETE(self): self._proxy()
    def log_message(self, *args): pass

server = http.server.HTTPServer(('0.0.0.0', 5001), Proxy)
server.serve_forever()
"""

with open("proxy.py", "w") as f:
    f.write(proxy_code)

subprocess.Popen(["python", "proxy.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Proxy started on port 5001...")
time.sleep(4)

# ── 4. Start Cloudflare tunnel ────────────────────────────────────
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# ── 5. Capture URL from output ────────────────────────────────────
public_url = None
print("Waiting for Cloudflare tunnel...")
for line in proc.stdout:
    line = line.decode()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group()
            break

if not public_url:
    print(" Could not extract tunnel URL.")
else:
    # ── 6. Wait for DNS to propagate before printing ──────────────
    print(f"Tunnel established. Waiting for DNS to propagate...")
    time.sleep(20)  # Cloudflare DNS takes 15-20s to go live

    # ── 7. Verify tunnel is actually reachable ────────────────────
    for attempt in range(10):
        try:
            r = requests.get(public_url, timeout=10)
            print(f"\n MLflow UI is live: {public_url}\n")
            break
        except Exception as e:
            print(f"Not reachable yet (attempt {attempt+1}/10): {e}")
            time.sleep(10)
    else:
        print(f"\n Tunnel URL printed but may still be propagating: {public_url}")
        print("Wait 30 more seconds then try opening it manually.")

In [ ]:
!mlflow server --backend-store-uri sqlite:///mlflow.db --port 5001 &

## Using ngrok

In [ ]:
'''import subprocess
import threading
import time
from pyngrok import ngrok

def run_mlflow_ui():
    # Start MLflow UI in a background process
    print("Starting MLflow UI...")
    mlflow_process = subprocess.Popen(["mlflow", "ui", "--host", "0.0.0.0"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("MLflow UI process started. Waiting for it to become available...")
    # Give MLflow UI some time to start
    time.sleep(5)
    for _ in range(30):
        if mlflow_process.poll() is not None: # If process exited, something went wrong
            stdout, stderr = mlflow_process.communicate()
            print(f"MLflow UI process exited with code {mlflow_process.returncode}")
            print(f"STDOUT: {stdout.decode()}")
            print(f"STDERR: {stderr.decode()}")
            raise RuntimeError("MLflow UI failed to start.")
        print("Checking for MLflow UI availability...")
        # More robust check could involve trying to connect to the port
        time.sleep(2)
    print("MLflow UI should be running locally now.")
    return mlflow_process

# Run MLflow UI in a separate thread
mlflow_thread = threading.Thread(target=run_mlflow_ui)
mlflow_thread.daemon = True
mlflow_thread.start()

# Wait for MLflow UI to start (adjust sleep time if necessary)
time.sleep(10)

# Create a public URL for the MLflow UI using ngrok
print("Creating ngrok tunnel...")
# The default MLflow UI port is 5000
public_url = ngrok.connect(5000)
print(f"MLflow UI is available at: {public_url}")'''

Using ngronk for MLflow. I need to upgrade to tier model to avoid the http and https handshake problem

In [ ]:
'''import subprocess, requests, time

for attempt in range(10):
    subprocess.run("pkill ngrok", shell=True)
    time.sleep(2)

    subprocess.Popen(
        ["ngrok", "http", "--scheme=https", "5000"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(5)

    try:
        resp = requests.get("http://localhost:4040/api/tunnels")
        tunnels = resp.json()["tunnels"]
        https_tunnels = [t for t in tunnels if t["public_url"].startswith("https://")]

        if https_tunnels:
            url = https_tunnels[0]["public_url"]
            if ".dev" not in url:  # Reject .dev domains
                print(f" Good domain: {url}")
                break
            else:
                print(f"Got .dev domain ({url}), retrying...")
    except Exception as e:
        print(f"Attempt {attempt+1}: {e}")
else:
    print("Could not get a non-.dev domain after 10 attempts.")'''